In [1]:
#SETUP
import pandas as pd
import numpy as np
import os

DATA = r'C:\Users\Sobia Khan\Downloads\DATA-adni\ADNIMERGE_CSVs'

# Load what we need
master  = pd.read_csv(os.path.join(DATA, 'master_baseline.csv'))
dx      = pd.read_csv(os.path.join(DATA, 'DXSUM.csv'), low_memory=False)
registry= pd.read_csv(os.path.join(DATA, 'REGISTRY.csv'), low_memory=False)

print(f"Master: {master.shape}")
print(f"DXSUM: {dx.shape}")
print(f"REGISTRY: {registry.shape}")
print(f"\nDXSUM columns: {dx.columns.tolist()[:15]}")
print(f"\nDXSUM VISCODE sample: {dx['VISCODE2'].unique()[:15]}")

Master: (1663, 23)
DXSUM: (15881, 42)
REGISTRY: (28858, 27)

DXSUM columns: ['ORIGPROT', 'COLPROT', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE', 'DIAGNOSIS', 'DXNORM', 'DXNODEP', 'DXMCI', 'DXMDES', 'DXMPTR1', 'DXMPTR2', 'DXMPTR3']

DXSUM VISCODE sample: ['bl' 'm06' 'm12' 'uns1' 'm36' 'm18' 'm24' 'm48' 'm60' 'm72' 'sc' 'm84'
 'm96' 'm108' 'm120']


In [4]:
#Build longitudinal diagnosis timeline 
# Map diagnosis codes to labels
# DIAGNOSIS is already string labels — no mapping needed
viscode_map = {
    'bl': 0, 'sc': 0, 'm06': 6, 'm12': 12,
    'm18': 18, 'm24': 24, 'm36': 36,
    'm48': 48, 'm60': 60, 'm72': 72,
    'm84': 84, 'm96': 96, 'm108': 108, 'm120': 120
}

# Clean DXSUM — use DIAGNOSIS directly, no mapping
dx_long = dx[['RID', 'VISCODE2', 'DIAGNOSIS']].copy()
dx_long = dx_long.dropna(subset=['DIAGNOSIS'])
dx_long['MONTHS'] = dx_long['VISCODE2'].map(viscode_map)
dx_long = dx_long.dropna(subset=['MONTHS'])
dx_long = dx_long.rename(columns={'DIAGNOSIS': 'DX_LABEL'})
dx_long = dx_long.sort_values(['RID', 'MONTHS'])

print(f"Longitudinal DX records: {dx_long.shape}")
print(f"\nDX_LABEL counts:")
print(dx_long['DX_LABEL'].value_counts())

# Sample subject trajectory
sample_rid = dx_long[dx_long['DX_LABEL']=='MCI']['RID'].iloc[0]
print(f"\nSample MCI subject (RID={sample_rid}) trajectory:")
print(dx_long[dx_long['RID']==sample_rid][['RID','MONTHS','DX_LABEL']])

Longitudinal DX records: (14662, 4)

DX_LABEL counts:
DX_LABEL
MCI         6179
CN          5644
Dementia    2839
Name: count, dtype: int64

Sample MCI subject (RID=2.0) trajectory:
      RID  MONTHS DX_LABEL
0     2.0     0.0       CN
166   2.0     6.0       CN
2861  2.0    36.0       CN
3907  2.0    60.0       CN
4701  2.0    72.0       CN
6465  2.0    84.0      MCI
7949  2.0    96.0       CN
9453  2.0   120.0       CN


In [5]:
#Define MCI converters 
# Focus on MCI subjects at baseline
mci_rids = master[master['DX_bl'] == 'MCI']['RID'].values
print(f"MCI subjects at baseline: {len(mci_rids)}")

# For each MCI subject, find if they ever convert to Dementia
converter_labels = []

for rid in mci_rids:
    subj_dx = dx_long[dx_long['RID'] == rid].sort_values('MONTHS')
    
    # Get all follow-up diagnoses after baseline
    followups = subj_dx[subj_dx['MONTHS'] > 0]
    
    if len(followups) == 0:
        # No follow-up — exclude
        continue
    
    # Check if ever converted to Dementia
    converted = (followups['DX_LABEL'] == 'Dementia').any()
    
    if converted:
        # Find time of first conversion
        conv_visit = followups[followups['DX_LABEL']=='Dementia'].iloc[0]
        conv_month = conv_visit['MONTHS']
    else:
        conv_month = None
    
    converter_labels.append({
        'RID': rid,
        'CONVERTER': int(converted),
        'CONV_MONTH': conv_month,
        'N_VISITS': len(subj_dx),
        'LAST_VISIT_MONTH': subj_dx['MONTHS'].max()
    })

conv_df = pd.DataFrame(converter_labels)
print(f"\nMCI subjects with follow-up: {len(conv_df)}")
print(f"\nConverter breakdown:")
print(conv_df['CONVERTER'].value_counts())
print(f"\nConversion rate: {conv_df['CONVERTER'].mean()*100:.1f}%")

MCI subjects at baseline: 767

MCI subjects with follow-up: 675

Converter breakdown:
CONVERTER
0    404
1    271
Name: count, dtype: int64

Conversion rate: 40.1%


In [6]:
#Define time-windowed converters (24-month):
# For clinical relevance — predict conversion within 24 months
conv_df['CONV_24M'] = (
    (conv_df['CONVERTER'] == 1) & 
    (conv_df['CONV_MONTH'] <= 24)
).astype(int)

conv_df['CONV_36M'] = (
    (conv_df['CONVERTER'] == 1) & 
    (conv_df['CONV_MONTH'] <= 36)
).astype(int)

print("Conversion windows:")
print(f"  Any conversion:      {conv_df['CONVERTER'].sum()} ({conv_df['CONVERTER'].mean()*100:.1f}%)")
print(f"  Within 24 months:    {conv_df['CONV_24M'].sum()} ({conv_df['CONV_24M'].mean()*100:.1f}%)")
print(f"  Within 36 months:    {conv_df['CONV_36M'].sum()} ({conv_df['CONV_36M'].mean()*100:.1f}%)")

# Distribution of conversion timing
converted_only = conv_df[conv_df['CONVERTER']==1]
print(f"\nConversion timing (months):")
print(converted_only['CONV_MONTH'].describe().round(1))

Conversion windows:
  Any conversion:      271 (40.1%)
  Within 24 months:    166 (24.6%)
  Within 36 months:    205 (30.4%)

Conversion timing (months):
count    271.0
mean      31.5
std       24.5
min        6.0
25%       12.0
50%       24.0
75%       36.0
max      120.0
Name: CONV_MONTH, dtype: float64


In [8]:
# Merge labels with master and save:
# Merge conversion labels with master baseline table
mci_master = master[master['DX_bl'] == 'MCI'].merge(
    conv_df[['RID','CONVERTER','CONV_24M','CONV_36M','CONV_MONTH',
             'N_VISITS','LAST_VISIT_MONTH']],
    on='RID', how='inner')

print(f"Final MCI dataset with labels: {mci_master.shape}")
print(f"\nConverter breakdown in final dataset:")
print(mci_master['CONVERTER'].value_counts())
print(f"\nFeature columns available:")
print(mci_master.columns.tolist())

# Save
mci_master.to_csv(os.path.join(DATA, 'mci_with_labels.csv'), index=False)
print(f"\n✅ Saved: mci_with_labels.csv")


Final MCI dataset with labels: (675, 29)

Converter breakdown in final dataset:
CONVERTER
0    404
1    271
Name: count, dtype: int64

Feature columns available:
['RID', 'VISCODE', 'SITEID', 'DX_bl', 'APOE4', 'APOE4_count', 'MMSCORE', 'CDGLOBAL', 'CDRSB', 'FAQTOTAL', 'TOTAL13', 'PTEDUCAT', 'PTGENDER', 'PTETHCAT', 'PTRACCAT', 'FIELD_STRENGTH', 'ST29SV', 'ST88SV', 'ST40TS', 'ST99TS', 'ST101SV', 'HIPPO_TOTAL', 'HIPPO_ICV', 'CONVERTER', 'CONV_24M', 'CONV_36M', 'CONV_MONTH', 'N_VISITS', 'LAST_VISIT_MONTH']

✅ Saved: mci_with_labels.csv


In [9]:
#Visualize converter lables 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

mci = pd.read_csv(os.path.join(DATA, 'mci_with_labels.csv'))

fig = make_subplots(rows=1, cols=3,
    subplot_titles=[
        'Converter vs Non-Converter',
        'Conversion Timing Distribution', 
        'APOE4 Rate by Conversion Status'
    ])

# 1. Converter pie/bar
counts = mci['CONVERTER'].value_counts().reindex([0,1])
fig.add_trace(go.Bar(
    x=['Non-Converter', 'Converter'],
    y=counts.values,
    marker_color=['#2ecc71','#e74c3c'],
    text=counts.values, textposition='inside',
    textfont=dict(color='white', size=14),
    showlegend=False
), row=1, col=1)
fig.update_yaxes(range=[0, counts.max()*1.15], row=1, col=1)

# 2. Conversion timing histogram
conv_times = mci[mci['CONVERTER']==1]['CONV_MONTH'].dropna()
fig.add_trace(go.Histogram(
    x=conv_times, nbinsx=15,
    marker_color='#e74c3c', opacity=0.8,
    showlegend=False,
    name='Conversion month'
), row=1, col=2)
fig.add_vline(x=24, line_dash='dash', line_color='black',
              annotation_text='24M', row=1, col=2)
fig.add_vline(x=36, line_dash='dash', line_color='orange',
              annotation_text='36M', row=1, col=2)

# 3. APOE4 rate by conversion status
apoe_rate = mci.groupby('CONVERTER')['APOE4'].mean() * 100
fig.add_trace(go.Bar(
    x=['Non-Converter', 'Converter'],
    y=[apoe_rate[0], apoe_rate[1]],
    marker_color=['#2ecc71','#e74c3c'],
    text=[f'{apoe_rate[0]:.1f}%', f'{apoe_rate[1]:.1f}%'],
    textposition='inside',
    textfont=dict(color='white', size=14),
    showlegend=False
), row=1, col=3)
fig.update_yaxes(range=[0, 85], row=1, col=3)

fig.update_layout(
    height=450, width=1100,
    title_text='MCI Converter Labels — Direction A Target Definition',
    title_x=0.5, title_font_size=15,
    paper_bgcolor='white', plot_bgcolor='white'
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='#eeeeee')

fig.write_html('02_converter_labels.html')
print("✅ Saved: 02_converter_labels.html")

# Print key stats
print(f"\nKey stats for DAAD application:")
print(f"  Total MCI with follow-up: {len(mci)}")
print(f"  Converters: {mci['CONVERTER'].sum()} ({mci['CONVERTER'].mean()*100:.1f}%)")
print(f"  24M converters: {mci['CONV_24M'].sum()} ({mci['CONV_24M'].mean()*100:.1f}%)")
print(f"  APOE4 in converters:     {mci[mci['CONVERTER']==1]['APOE4'].mean()*100:.1f}%")
print(f"  APOE4 in non-converters: {mci[mci['CONVERTER']==0]['APOE4'].mean()*100:.1f}%")
fig.show()

✅ Saved: 02_converter_labels.html

Key stats for DAAD application:
  Total MCI with follow-up: 675
  Converters: 271 (40.1%)
  24M converters: 166 (24.6%)
  APOE4 in converters:     62.7%
  APOE4 in non-converters: 39.7%


c:\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.


